<a href="https://colab.research.google.com/github/diwashsapkota/EMDC-Training-Contents/blob/main/Transcript_from_YouTube_Audio_Files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Extract Transcript from YouTube / Audio Files**

Model Card: https://github.com/openai/whisper/blob/main/model-card.md

Notes:
- Input files (mp4, mp3, wav) → from Drive or YouTube URL  
- Output files: .txt, .srt, .vtt, .tsv  
- For .txt, there are no timestamps; other formats include timestamps  
- GPU is required  
- If GPU overload occurs, please *Disconnect and delete runtime*  
- The process may take some time, so please be patient  

Modes:
- tiny: audio recordings without noise (voice notes)  
- small: two-way audio recordings (interviews)  
- medium: Instagram Live recordings, seminars  
- large: noisy audio recordings (sermons), online classes, KDL, etc.

## Important Notes Before Running:

*   **"Run Anyway" Warning:** If you see a warning about running a notebook from an untrusted source, select "Run Anyway" to proceed. This is common for Colab notebooks from GitHub and ensures all necessary setup steps can execute.
*   **Google Drive Permissions:** When you run STEP 1, you will be prompted to grant Google Drive access. This is necessary to save your output files to your Drive and to access any audio files you might want to transcribe from Drive in STEP 2.
*   **Whisper Model Details:**
    *   `tiny`: Very fast, smallest model, but less accurate. Best for clear audio with single speakers and minimal background noise.
    *   `base`: A good balance of speed and accuracy for general purposes.
    *   `small`: More accurate than `base`, suitable for two-way conversations like interviews.
    *   `medium`: Higher accuracy, good for recordings with some background noise, like seminars or live streams.
    *   `large` / `large-v1` / `large-v2` / `large-v3`: The most accurate and largest models. Ideal for very noisy environments, complex audio (e.g., sermons, online classes with multiple speakers), and when high transcription quality is paramount. `large-v2` and `large-v3` are generally improved versions over `large-v1`.
    *   `turbo`: A faster, distilled version of the `large-v2` model, offering a good trade-off between speed and accuracy for real-time applications where `large-v2` might be too slow.

Choose the model that best fits the quality and complexity of your audio to balance transcription time and accuracy.

In [ ]:
# @title STEP 1 - Install Required Libraries
!pip install -q yt-dlp
!pip install -q openai
!pip install -q cohere
!pip install -q git+https://github.com/openai/whisper.git

import os, re
import torch
from pathlib import Path

import whisper
from whisper.utils import get_writer
from google.colab import drive

whisper_model = "turbo" # @param ["tiny", "base", "small", "medium", "large", "large-v1", "large-v2", "large-v3", "turbo"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model(whisper_model).to(DEVICE)

drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title STEP 2 - Input Data
data_type = "youtube url" # @param ["youtube url", "audio file (drive)", "custom"]
path_or_link = "https://www.youtube.com/watch?v=SUqQrlLV0tU" # @param {type:"string"}
save_output_to = "/content/drive/MyDrive/Whisper_Transcripts/" # @param {type:"string"}
audio_extension = "mp4" # @param {type: "string"}
text = True # @param {type:"boolean"}
srt = True # @param {type:"boolean"}
vtt = False # @param {type:"boolean"}
tsv = False # @param {type:"boolean"}
auto_download = True # @param {type:"boolean"}

def to_snake_case(name):
    return name.lower().replace(" ", "_").replace(":", "_").replace("__", "_").replace("/","_")

# Removed download_youtube_audio function, now handled by STEP 3.

### Output Location and Download:

*   The `save_output_to` variable determines where your transcription files (.txt, .srt, etc.) will be saved. By default, it's set to `/content/drive/MyDrive/Whisper_Transcripts/` so your outputs are easily accessible in your Google Drive.
*   You can change this path if you prefer a different location. If left empty, files will be saved next to the original audio file.
*   The `auto_download` option, if checked, will automatically download the generated transcription files to your local machine after processing each file. Otherwise, you can find them in the specified output directory in Google Drive (via the folder icon on the left pane).



### Language Selection

You can specify the language of the audio to be transcribed. Choosing 'Auto Detect' will allow Whisper to automatically identify the language. If you know the language, selecting it can improve accuracy and speed.

In [ ]:
# @title Choose Transcription Language

import ipywidgets as widgets
from IPython.display import display

language_options = [
    "Auto Detect", "English", "Chinese", "German", "Spanish", "Russian",
    "Korean", "French", "Japanese", "Portuguese", "Turkish", "Polish",
    "Catalan", "Dutch", "Arabic", "Swedish", "Italian", "Indonesian",
    "Hindi", "Finnish", "Vietnamese", "Hebrew", "Ukrainian", "Greek",
    "Malay", "Czech", "Romanian", "Danish", "Hungarian", "Tamil",
    "Norwegian", "Thai", "Urdu", "Persian", "Myanmar", "Latvian",
    "Belarusian", "Bulgarian", "Estonian", "Azerbaijani", "Serbian",
    "Slovak", "Slovenian", "Albanian", "Kazakh", "Lithuanian",
    "Macedonian", "Mongolian", "Welsh", "Georgian"
]

language_dropdown = widgets.Dropdown(
    options=language_options,
    value="Auto Detect",
    description="Language:",
    layout=widgets.Layout(width="auto")
)

display(language_dropdown)

def get_selected_transcribe_language():
    selected = language_dropdown.value
    if selected == "Auto Detect":
        return None
    return selected

Dropdown(description='Language:', layout=Layout(width='auto'), options=('Auto Detect', 'English', 'Chinese', '…

In [ ]:
import os
import re
import copy
import subprocess # Required for yt-dlp
from pathlib import Path
from whisper.utils import get_writer
from google.colab import files


# ============================================================
# SETTINGS PART 3
# ============================================================

# True  = skip file kalau output sesuai checklist sudah ada
# False = tetap transcribe ulang walaupun output sudah ada
skip_existing = True # @param {type:"boolean"}

# Kalau Part 2 belum punya save_output_to, Part 3 akan pakai ini.
# Kosong "" = simpan hasil transcript di folder yang sama dengan file audio/video.
try:
    save_output_to
except NameError:
    save_output_to = ""


# ============================================================
# Helper: YouTube ID extractor
# ============================================================

def extract_youtube_id(url):
    pattern = r'(?:https?://)?(?:www\.)?(?:youtube\.com/(?:[^/\n\s]+/\S+/|(?:v|e(?:mbed)?)/|\S*?[?&]v=))([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)

    if match:
        return match.group(1)

    return None


# ============================================================
# Helper: YouTube audio downloader using yt-dlp with cookie handling
# ============================================================

def download_youtube_audio_with_ytdlp(youtube_url, output_path=".", cookies_file=None):
    youtube_id = extract_youtube_id(youtube_url)
    if not youtube_id:
        print(f"Could not extract YouTube ID from URL: {youtube_url}")
        return None

    # Define the output template for yt-dlp
    output_template = os.path.join(output_path, f"{youtube_id}.%(ext)s")

    cmd = [
        "yt-dlp",
        "-x",  # Extract audio
        "--audio-format", "mp3",
        "--audio-quality", "0", # Best quality
        "-o", output_template,
        youtube_url
    ]

    if cookies_file and os.path.exists(cookies_file):
        cmd.extend(["--cookies", cookies_file])
        print(f"Attempting download with cookies from: {cookies_file}")
    else:
        print("Attempting download without cookies.")

    try:
        process = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print("yt-dlp stdout:\n", process.stdout)
        # yt-dlp often renames the file to .mp3, so we need to find it.
        # A simple approach is to list files in output_path and find the one that starts with youtube_id
        for f in os.listdir(output_path):
            if f.startswith(youtube_id) and f.endswith(".mp3"):
                full_path = os.path.join(output_path, f)
                print(f"Audio downloaded successfully to: {full_path}")
                return full_path
        print("yt-dlp finished, but could not find the downloaded MP3 file.")
        return None
    except subprocess.CalledProcessError as e:
        print(f"Error downloading audio with yt-dlp: {e}")
        print("yt-dlp stderr:\n", e.stderr)
        return None
    except Exception as e:
        print(f"An unexpected error occurred during yt-dlp download: {e}")
        return None

def upload_cookies_file():
    print("Please upload your 'cookies.txt' file for YouTube authentication:")
    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded.")
        return None

    filename = next(iter(uploaded))
    with open(filename, "wb") as f:
        f.write(uploaded[filename])

    print(f"Cookies file '{filename}' uploaded successfully.")
    return filename


# ============================================================
# Helper: Resolve Google Drive path
# ============================================================

def resolve_drive_path(path_value):
    raw_path = str(path_value).strip()

    if raw_path == "":
        return ""

    if raw_path.startswith("/content/drive/"):
        full_path = raw_path

    elif raw_path.startswith("Shareddrives/"):
        full_path = os.path.join("/content/drive", raw_path)

    elif raw_path.startswith("/Shareddrives/"):
        full_path = os.path.join("/content/drive", raw_path.lstrip("/"))

    elif raw_path.startswith("MyDrive/"):
        full_path = os.path.join("/content/drive", raw_path)

    elif raw_path.startswith("/MyDrive/"):
        full_path = os.path.join("/content/drive", raw_path.lstrip("/"))

    else:
        full_path = os.path.join("/content/drive/MyDrive", raw_path.lstrip("/"))

    return os.path.normpath(full_path)


# ============================================================
# Helper: Normalize extension
# ============================================================

def normalize_extension(extension):
    extension = str(extension).strip().lower()

    if extension == "":
        return ""

    if not extension.startswith("."):
        extension = "." + extension

    return extension


# ============================================================
# Helper: Output path + skip existing
# ============================================================

def get_output_directory_for_file(file_path, output_directory=None):
    file_path = Path(file_path)

    if output_directory is None or str(output_directory).strip() == "":
        return file_path.parent

    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)

    return output_directory


def get_expected_output_files(file_path, output_directory, text, srt, vtt, tsv):
    file_path = Path(file_path)
    output_directory = Path(output_directory)
    original_stem = file_path.stem

    expected_files = []

    if text:
        expected_files.append(output_directory / f"{original_stem}.txt")

    if srt:
        expected_files.append(output_directory / f"{original_stem}.srt")

    if vtt:
        expected_files.append(output_directory / f"{original_stem}.vtt")

    if tsv:
        expected_files.append(output_directory / f"{original_stem}.tsv")

    return expected_files


def file_has_content(file_path):
    file_path = Path(file_path)

    return (
        file_path.exists()
        and file_path.is_file()
        and file_path.stat().st_size > 0
    )


def should_skip_transcription(
    file_path,
    output_directory,
    text,
    srt,
    vtt,
    tsv,
    skip_existing=True
):
    if not skip_existing:
        return False

    expected_files = get_expected_output_files(
        file_path=file_path,
        output_directory=output_directory,
        text=text,
        srt=srt,
        vtt=vtt,
        tsv=tsv
    )

    if not expected_files:
        return False

    # Skip hanya kalau SEMUA output yang dicentang sudah ada dan tidak kosong.
    return all(file_has_content(output_file) for output_file in expected_files)


def print_existing_outputs(file_path, output_directory, text, srt, vtt, tsv):
    expected_files = get_expected_output_files(
        file_path=file_path,
        output_directory=output_directory,
        text=text,
        srt=srt,
        vtt=vtt,
        tsv=tsv
    )

    print("Existing output found. Skipping this file:")
    for output_file in expected_files:
        status = "OK" if file_has_content(output_file) else "MISSING"
        print(f"- [{status}] {output_file}")


# ============================================================
# Helper: Clean duplicate transcription text
# ============================================================

def normalize_for_compare(text):
    text = str(text).strip().lower()
    text = re.sub(r"[^\w\s']", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def clean_segment_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()

    if text == "":
        return text

    # Amen Amen Amen -> Amen
    text = re.sub(
        r"\b(\w+)(?:\s+\1\b){2,}",
        r"\1",
        text,
        flags=re.IGNORECASE
    )

    # thank you thank you thank you -> thank you
    for phrase_len in range(2, 6):
        words = text.split()

        if len(words) >= phrase_len * 3:
            rebuilt = []
            i = 0

            while i < len(words):
                phrase = words[i:i + phrase_len]
                repeat_count = 1
                j = i + phrase_len

                while j + phrase_len <= len(words):
                    next_phrase = words[j:j + phrase_len]

                    phrase_clean = [
                        w.lower().strip(".,!?;:")
                        for w in phrase
                    ]
                    next_phrase_clean = [
                        w.lower().strip(".,!?;:")
                        for w in next_phrase
                    ]

                    if next_phrase_clean == phrase_clean:
                        repeat_count += 1
                        j += phrase_len
                    else:
                        break

                rebuilt.extend(phrase)
                i += phrase_len * repeat_count

            text = " ".join(rebuilt)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_whisper_result(result):
    cleaned_result = copy.deepcopy(result)

    if "segments" not in cleaned_result or not cleaned_result["segments"]:
        if "text" in cleaned_result:
            cleaned_result["text"] = clean_segment_text(cleaned_result["text"])
        return cleaned_result

    cleaned_segments = []
    previous_compare_text = ""
    previous_end = None
    removed_count = 0

    for segment in cleaned_result["segments"]:
        segment_text = segment.get("text", "")
        cleaned_text = clean_segment_text(segment_text)
        compare_text = normalize_for_compare(cleaned_text)

        if compare_text == "":
            removed_count += 1
            continue

        start_time = float(segment.get("start", 0))
        end_time = float(segment.get("end", start_time))
        duration = max(0, end_time - start_time)

        if previous_end is None:
            gap = 999
        else:
            gap = start_time - previous_end

        is_duplicate_segment = False

        # Hapus duplicate pendek yang berurutan, contoh:
        # Amen.
        # Amen.
        # Amen.
        if (
            compare_text == previous_compare_text
            and duration <= 4.0
            and gap <= 6.0
            and len(compare_text.split()) <= 8
        ):
            is_duplicate_segment = True

        # Hapus duplicate exact yang sangat dekat.
        if (
            compare_text == previous_compare_text
            and gap <= 2.0
        ):
            is_duplicate_segment = True

        if is_duplicate_segment:
            removed_count += 1
            continue

        segment["text"] = " " + cleaned_text
        cleaned_segments.append(segment)

        previous_compare_text = compare_text
        previous_end = end_time

    cleaned_result["segments"] = cleaned_segments
    cleaned_result["text"] = " ".join(
        segment.get("text", "").strip()
        for segment in cleaned_segments
        if segment.get("text", "").strip()
    )

    print(f"\nCleaned transcript: removed {removed_count} duplicate/empty segment(s).")

    return cleaned_result


# ============================================================
# Main transcription function for single file
# ============================================================

def transcribe_file(
    model,
    file,
    text,
    srt,
    vtt,
    tsv,
    auto_download,
    output_directory=None,
    skip_existing=True,
    language=None
):
    file_path = Path(file)

    print("\n==================================================")
    print(f"Checking file: {file_path}")
    print("==================================================")

    if not file_path.exists():
        raise FileNotFoundError(f"File not found:\n{file_path}")

    output_directory = get_output_directory_for_file(
        file_path=file_path,
        output_directory=output_directory
    )

    print(f"Saving transcript to: {output_directory}")

    if should_skip_transcription(
        file_path=file_path,
        output_directory=output_directory,
        text=text,
        srt=srt,
        vtt=vtt,
        tsv=tsv,
        skip_existing=skip_existing
    ):
        print_existing_outputs(
            file_path=file_path,
            output_directory=output_directory,
            text=text,
            srt=srt,
            vtt=vtt,
            tsv=tsv
        )

        return {
            "skipped": True,
            "file": str(file_path),
            "output_directory": str(output_directory)
        }

    print(f"\nTranscribing file: {file_path}")

    transcribe_options = {"verbose": True}
    if language:
        transcribe_options["language"] = language

    raw_result = model.transcribe(str(file_path), **transcribe_options)

    result = clean_whisper_result(raw_result)

    original_stem = file_path.stem

    if text:
        txt_path = output_directory / f"{original_stem}.txt"
        print(f"\nCreating clean text file: {txt_path}")

        with open(txt_path, "w", encoding="utf-8") as txt:
            txt.write(result["text"])

    if srt:
        srt_path = output_directory / f"{original_stem}.srt"
        print(f"\nCreating clean SRT file: {srt_path}")

        srt_writer = get_writer("srt", str(output_directory))
        srt_writer(
            result,
            original_stem,
            {
                "max_line_width": None,
                "max_line_count": None,
                "highlight_words": False
            }
        )

    if vtt:
        vtt_path = output_directory / f"{original_stem}.vtt"
        print(f"\nCreating clean VTT file: {vtt_path}")

        vtt_writer = get_writer("vtt", str(output_directory))
        vtt_writer(
            result,
            original_stem,
            {
                "max_line_width": None,
                "max_line_count": None,
                "highlight_words": False
            }
        )

    if tsv:
        tsv_path = output_directory / f"{original_stem}.tsv"
        print(f"\nCreating clean TSV file: {tsv_path}")

        tsv_writer = get_writer("tsv", str(output_directory))
        tsv_writer(
            result,
            original_stem,
            {
                "max_line_width": None,
                "max_line_count": None,
                "highlight_words": False
            }
        )

    if auto_download:
        for output_file in output_directory.glob(f"{original_stem}*"):
            if output_file.suffix.lower() in [".txt", ".srt", ".vtt", ".tsv"]:
                print(f"Downloading {output_file}")
                files.download(str(output_file))

    result["skipped"] = False
    result["file"] = str(file_path)
    result["output_directory"] = str(output_directory)

    return result


# ============================================================
# Main transcription function for folder
# ============================================================

def transcribe_folder(
    model,
    folder_path,
    audio_extension,
    text,
    srt,
    vtt,
    tsv,
    auto_download,
    output_directory=None,
    skip_existing=True,
    language=None
):
    folder_path = Path(folder_path)
    audio_extension = normalize_extension(audio_extension)

    if not folder_path.exists():
        raise FileNotFoundError(f"Folder not found:\n{folder_path}")

    if not folder_path.is_dir():
        raise NotADirectoryError(f"This is not a folder:\n{folder_path}")

    if audio_extension == "":
        raise ValueError(
            "Because path_or_link is a folder, audio_extension must be filled.\n"
            "Example: mp4, mp3, m4a, wav"
        )

    if output_directory is not None and str(output_directory).strip() == "":
        output_directory = Path(output_directory)
        output_directory.mkdir(parents=True, exist_ok=True)
    else:
        output_directory = None

    print("\n==================================================")
    print(f"Folder source: {folder_path}")
    print(f"File extension: {audio_extension}")

    if output_directory:
        print(f"Output folder: {output_directory}")
    else:
        print("Output folder: same as each source file folder")

    print(f"Skip existing: {skip_existing}")
    print("==================================================")

    files_to_transcribe = sorted(folder_path.glob(f"*{audio_extension}"))

    if not files_to_transcribe:
        raise FileNotFoundError(
            f"No files with extension {audio_extension} found in:\n{folder_path}"
        )

    print(f"\nFound {len(files_to_transcribe)} file(s):")
    for item in files_to_transcribe:
        print(f"- {item.name}")

    all_results = {}
    processed_count = 0
    skipped_count = 0
    failed_count = 0

    for index, media_file in enumerate(files_to_transcribe, start=1):
        print(f"\n\n========== Processing {index}/{len(files_to_transcribe)} ==========")

        try:
            result = transcribe_file(
                model=model,
                file=str(media_file),
                text=text,
                srt=srt,
                vtt=vtt,
                tsv=tsv,
                auto_download=auto_download,
                output_directory=output_directory,
                skip_existing=skip_existing,
                language=language # Pass language here
            )

            all_results[media_file.name] = result

            if isinstance(result, dict) and result.get("skipped") is True:
                skipped_count += 1
            else:
                processed_count += 1

        except Exception as e:
            print(f"\nFailed to transcribe {media_file.name}")
            print(f"Error: {e}")
            all_results[media_file.name] = None
            failed_count += 1

    print("\n==================================================")
    print("Finished transcribing folder.")
    print(f"Processed: {processed_count}")
    print(f"Skipped: {skipped_count}")
    print(f"Failed: {failed_count}")
    print("==================================================")

    return all_results


# ============================================================
# Main process
# ============================================================

if data_type == "youtube url":
    print(f"Downloading {path_or_link} using yt-dlp...")
    downloaded_audio = None
    cookies_path = None

    # First attempt without cookies
    downloaded_audio = download_youtube_audio_with_ytdlp(path_or_link, output_path=".")

    if not downloaded_audio:
        print("\nInitial download failed. Checking for cookie-related issues...")
        print("Attempting to retry with cookies.")

        # User uploads cookies.txt
        cookies_path = upload_cookies_file()

        if cookies_path:
            print(f"Retrying download with cookies from {cookies_path}...")
            downloaded_audio = download_youtube_audio_with_ytdlp(path_or_link, output_path=".", cookies_file=cookies_path)
        else:
            print("No cookies file provided. Cannot retry with cookies.")

    if downloaded_audio:
        audio = downloaded_audio # Actual file path
        if save_output_to:
            output_directory = Path(save_output_to)
        else:
            output_directory = None

        result = transcribe_file(
            model=model,
            file=audio,
            text=text,
            srt=srt,
            vtt=vtt,
            tsv=tsv,
            auto_download=auto_download,
            output_directory=output_directory,
            skip_existing=skip_existing,
            language=get_selected_transcribe_language() # Pass language here
        )
    else:
        print("Failed to download the audio using yt-dlp. Please check the URL and try again or try manual download.")

elif data_type == "audio file (drive)":
    from google.colab import drive

    drive.mount("/content/drive")

    full_drive_path = resolve_drive_path(path_or_link)

    if save_output_to:
        output_directory = resolve_drive_path(save_output_to)
    else:
        output_directory = None

    print("\nResolved input path:")
    print(full_drive_path)

    if output_directory:
        print("\nResolved output path:")
        print(output_directory)

    if not os.path.exists(full_drive_path):
        raise FileNotFoundError(
            f"File or folder not found:\n{full_drive_path}\n\n"
            "Check whether the path is in MyDrive or Shareddrives, and verify spelling/capitalization."
        )

    if os.path.isdir(full_drive_path):
        result = transcribe_folder(
            model=model,
            folder_path=full_drive_path,
            audio_extension=audio_extension,
            text=text,
            srt=srt,
            vtt=vtt,
            tsv=tsv,
            auto_download=auto_download,
            output_directory=output_directory,
            skip_existing=skip_existing,
            language=get_selected_transcribe_language() # Pass language here
        )

    else:
        result = transcribe_file(
            model=model,
            file=full_drive_path,
            text=text,
            srt=srt,
            vtt=vtt,
            tsv=tsv,
            auto_download=auto_download,
            output_directory=output_directory,
            skip_existing=skip_existing,
            language=get_selected_transcribe_language() # Pass language here
        )


else:
    if save_output_to:
        output_directory = save_output_to
    else:
        output_directory = None

    result = transcribe_file(
        model=model,
        file=path_or_link,
        text=text,
        srt=srt,
        vtt=vtt,
        tsv=tsv,
        auto_download=auto_download,
        output_directory=output_directory,
        skip_existing=skip_existing,
        language=get_selected_transcribe_language() # Pass language here
    )

Attempting download without cookies.
yt-dlp stdout:
 [youtube] Extracting URL: https://www.youtube.com/watch?v=SUqQrlLV0tU
[youtube] SUqQrlLV0tU: Downloading webpage
[youtube] SUqQrlLV0tU: Downloading android vr player API JSON
[info] SUqQrlLV0tU: Downloading 1 format(s): 251
[download] Destination: ./SUqQrlLV0tU.webm

[download]   0.0% of   21.99MiB at  721.91KiB/s ETA 00:31
[download]   0.0% of   21.99MiB at    1.59MiB/s ETA 00:13
[download]   0.0% of   21.99MiB at    3.09MiB/s ETA 00:07
[download]   0.1% of   21.99MiB at    5.52MiB/s ETA 00:03
[download]   0.1% of   21.99MiB at    4.15MiB/s ETA 00:05
[download]   0.3% of   21.99MiB at    4.77MiB/s ETA 00:04
[download]   0.6% of   21.99MiB at    6.56MiB/s ETA 00:03
[download]   1.1% of   21.99MiB at    9.47MiB/s ETA 00:02
[download]   2.3% of   21.99MiB at   15.16MiB/s ETA 00:01
[download]   4.5% of   21.99MiB at   25.04MiB/s ETA 00:00
[download]   9.1% of   21.99MiB at   41.77MiB/s ETA 00:00
[download]  18.2% of   21.99MiB at   69.5

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **GENERATE AUDIO**

Model Card: https://github.com/rany2/edge-tts

List Voices: https://github.com/FoloToy/folotoy-server-self-hosting/wiki/Edge%E2%80%90TTS%E2%80%90Voices

Notes:
- Input file (**txt**) -> upload, input file  
- Output file: .mp3  
- GPU is not required

## Generate Audio from Text

This section allows you to convert text into spoken audio using the Microsoft Edge Text-to-Speech (Edge-TTS) service. You can choose from a wide variety of voices, including different languages, genders, and speaking styles, as listed in the provided link.

*   **Model:** Edge-TTS leverages cloud-based text-to-speech capabilities, offering natural-sounding voices.
*   **Voices:** The list of available voices (`VOICE:` dropdown) gives you options for different accents, genders, and languages. You can select the one that best suits your needs.



In [ ]:
# @title STEP 1 - Install Library & Config
!pip -q install edge-tts ipywidgets nest_asyncio

import os
import asyncio
import ipywidgets as widgets
from IPython.display import display
from google.colab import files
import edge_tts
import nest_asyncio

nest_asyncio.apply()

OUT_DIR = "output_audio"
os.makedirs(OUT_DIR, exist_ok=True)


# =========================
# LOAD VOICES FROM edge-tts
# =========================
async def get_voices():
    voices = await edge_tts.list_voices()
    voices = sorted(voices, key=lambda v: v["ShortName"])
    return voices

VOICES = asyncio.get_event_loop().run_until_complete(get_voices())

voice_options = []
for v in VOICES:
    short = v.get("ShortName", "")
    gender = v.get("Gender", "")
    locale = v.get("Locale", "")
    label = f"{short} | {gender} | {locale}"
    voice_options.append((label, short))

# default voice
default_voice = "en-US-GuyNeural"
available_shortnames = [value for _, value in voice_options]
if default_voice not in available_shortnames:
    default_voice = available_shortnames[0]


# =========================
# UI: SELECT VOICE
# =========================
voice_dropdown = widgets.Dropdown(
    options=voice_options,
    value=default_voice,
    description="VOICE:",
    layout=widgets.Layout(width="700px")
)

display(voice_dropdown)

VOICE = voice_dropdown.value


def get_selected_voice():
    return voice_dropdown.value

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 78.5 MB/s eta 0:00:00


Dropdown(description='VOICE:', index=106, layout=Layout(width='700px'), options=(('af-ZA-AdriNeural | Female |…

In [ ]:
# @title STEP 2 - Upload 1 TXT -> Edit Manual -> Save & Continue
import ipywidgets as widgets
from IPython.display import display
import os

uploader = widgets.FileUpload(
    accept=".txt",
    multiple=False,
    description="Upload 1 TXT"
)

btn_load = widgets.Button(description="Load", button_style="info")
btn_save = widgets.Button(description="Save & Continue to Step 3", button_style="success")

status = widgets.Output()

text_editor = widgets.Textarea(
    value="(Upload 1 .txt file, lalu klik Load. Setelah itu edit manual di sini.)",
    description="TEXT:",
    layout=widgets.Layout(width="100%", height="320px")
)

display(uploader, widgets.HBox([btn_load, btn_save]), text_editor, status)

# Variables for Step 3
FINAL_TEXT = None
FINAL_TXT_PATH = None
SOURCE_NAME = None

def _get_uploaded_text_single():
    global SOURCE_NAME
    if not uploader.value:
        return None
    if len(uploader.value) != 1:
        return None
    item = next(iter(uploader.value.values()))
    SOURCE_NAME = item["metadata"]["name"]
    return item["content"].decode("utf-8", errors="replace")

def on_load(_):
    with status:
        status.clear_output()
        raw = _get_uploaded_text_single()
        if raw is None:
            print("Upload exactly 1 .txt file only.")
            return
        text_editor.value = raw
        print("Loaded:", SOURCE_NAME, "| chars:", len(raw))
        print("You can now edit the text manually.")

def on_save(_):
    global FINAL_TEXT, FINAL_TXT_PATH
    with status:
        status.clear_output()
        edited = text_editor.value.strip()
        if not edited:
            print("Text is empty. Please load or type your text first.")
            return

        FINAL_TEXT = edited

        base = "edited_text"
        if SOURCE_NAME:
            base = os.path.splitext(SOURCE_NAME)[0] + "_EDITED"

        out_dir = globals().get("OUT_DIR", "output_audio")
        os.makedirs(out_dir, exist_ok=True)

        FINAL_TXT_PATH = os.path.join(out_dir, base + ".txt")
        with open(FINAL_TXT_PATH, "w", encoding="utf-8") as f:
            f.write(FINAL_TEXT)

        print("✅ Saved for Step 3.")
        print("Edited TXT:", FINAL_TXT_PATH)
        print("Now continue by running STEP 3.")

btn_load.on_click(on_load)
btn_save.on_click(on_save)

FileUpload(value={}, accept='.txt', description='Upload 1 TXT')

Textarea(value='(Upload 1 .txt file, lalu klik Load. Setelah itu edit manual di sini.)', description='TEXT:', …

Output()

### Text Input for Audio Generation:

You can provide text for audio generation in two ways:

1.  **Upload a TXT file:** Use the `Upload 1 TXT` button to upload a `.txt` file from your local machine.
2.  **Directly type or paste:** After uploading, or if you don't upload a file, you can directly type or paste your desired text into the `TEXT:` editor below.

After inputting your text, click `Save & Continue to Step 3` to prepare it for audio generation.

In [ ]:
# @title STEP 3 - Generate & Download (without loading bar)
import ipywidgets as widgets
from IPython.display import display, Audio

rate = widgets.IntSlider(value=0, min=-50, max=50, step=5, description="Speed %")
pitch = widgets.IntSlider(value=0, min=-20, max=20, step=2, description="Pitch Hz")
out_name = widgets.Text(value="output.mp3", description="MP3:", layout=widgets.Layout(width="60%"))

btn_generate = widgets.Button(description="Generate Audio", button_style="warning")
btn_download = widgets.Button(description="Download MP3", button_style="success", disabled=True)

spinner = widgets.HTML(value="")
status = widgets.Output()
player = widgets.Output()

display(
    rate,
    pitch,
    out_name,
    widgets.HBox([btn_generate, btn_download]),
    spinner,
    status,
    player
)

LAST_MP3_PATH = None

async def tts_to_file(text, out_path, voice, rate_pct, pitch_hz):
    r = f"{rate_pct:+d}%"
    p = f"{pitch_hz:+d}Hz"
    comm = edge_tts.Communicate(text=text, voice=voice, rate=r, pitch=p)
    await comm.save(out_path)

def on_generate(_):
    global LAST_MP3_PATH

    with player:
        player.clear_output()
    with status:
        status.clear_output()

    if "FINAL_TEXT" not in globals() or not FINAL_TEXT:
        with status:
            print("FINAL_TEXT is not available yet. Go back to Step 2, then click 'Save & Continue to Step 3'.")
        return

    name = out_name.value.strip() or "output.mp3"
    if not name.lower().endswith(".mp3"):
        name += ".mp3"

    out_path = os.path.join(OUT_DIR, name)

    selected_voice = get_selected_voice() if "get_selected_voice" in globals() else VOICE

    btn_generate.disabled = True
    btn_download.disabled = True
    btn_generate.description = "Generating..."
    spinner.value = "<b>⏳ Generating audio...</b>"

    try:
        loop = asyncio.get_event_loop()
        loop.run_until_complete(
            tts_to_file(FINAL_TEXT, out_path, selected_voice, rate.value, pitch.value)
        )

        LAST_MP3_PATH = out_path
        btn_download.disabled = False

        with status:
            print("✅ Audio generated successfully:")
            print(out_path)
            print("Voice:", selected_voice)

        with player:
            display(Audio(out_path))

    except Exception as e:
        with status:
            print("❌ Failed to generate audio:", repr(e))

    finally:
        btn_generate.disabled = False
        btn_generate.description = "Generate Audio"
        spinner.value = ""

def on_download(_):
    with status:
        status.clear_output()

    if not LAST_MP3_PATH or not os.path.exists(LAST_MP3_PATH):
        with status:
            print("No MP3 file available yet. Click 'Generate Audio' first.")
        return

    files.download(LAST_MP3_PATH)

btn_generate.on_click(on_generate)
btn_download.on_click(on_download)

IntSlider(value=0, description='Speed %', max=50, min=-50, step=5)

IntSlider(value=0, description='Pitch Hz', max=20, min=-20, step=2)

Text(value='output.mp3', description='MP3:', layout=Layout(width='60%'))

HTML(value='')

Output()

Output()